In [1]:
import os

import pandas as pd

from framework.dataset_specification import NamedDatasetSpecifications, DatasetSpecification
from framework.enumerations import EvaluationDatasetSampling
from framework.flow_transformer_binary_classification import FlowTransformer
from framework.flow_transformer_parameters import FlowTransformerParameters
from implementations.classification_heads import *
from implementations.input_encodings import *
from implementations.pre_processings import StandardPreProcessing
from implementations.transformers.basic_transformers import BasicTransformer
from implementations.transformers.named_transformers import *

2026-04-12 13:49:50.177797: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2026-04-12 13:49:50.177828: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


In [2]:
encodings = [
    NoInputEncoder(),
    RecordLevelEmbed(64),
    CategoricalFeatureEmbed(EmbedLayerType.Dense, 16),
    CategoricalFeatureEmbed(EmbedLayerType.Lookup, 16),
    CategoricalFeatureEmbed(EmbedLayerType.Projection, 16),
    RecordLevelEmbed(64, project=True)
]

classification_heads = [
    LastTokenClassificationHead(),
    FlattenClassificationHead(),
    GlobalAveragePoolingClassificationHead(),
    CLSTokenClassificationHead(),
    FeaturewiseEmbedding(project=False),
    FeaturewiseEmbedding(project=True),
]

transformers = [
    BasicTransformer(2, 128, n_heads=2),
    BasicTransformer(2, 128, n_heads=2, is_decoder=True),
    GPTSmallTransformer(),
    BERTSmallTransformer()
]

In [3]:
custom_dataset_spec = DatasetSpecification(
    include_fields=['L4_SRC_PORT', 'L4_DST_PORT', 'PROTOCOL', 'FLOW_DURATION_MILLISECONDS', 'OUT_PKTS', 'OUT_BYTES', 'IN_PKTS', 'IN_BYTES', 'L7_PROTO', 'TCP_FLAGS'],
    categorical_fields=['L4_SRC_PORT', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO', 'TCP_FLAGS'],
    class_column="Attack",
    benign_label="Benign"
)

datasets = [
    ("Balanced_Dataset_Binary", r"d:\Workplace\vscode\ai-ids-master\FlowTransformer\data\balanced_dataset_tcp_udp.csv", custom_dataset_spec, 0.2, EvaluationDatasetSampling.RandomRows)
]

In [4]:
pre_processing = StandardPreProcessing(n_categorical_levels=32)

# Define the transformer
ft = FlowTransformer(pre_processing=pre_processing,
                     input_encoding=encodings[0],
                     sequential_model=transformers[0],
                     classification_head=classification_heads[0],
                     params=FlowTransformerParameters(window_size=8, mlp_layer_sizes=[128], mlp_dropout=0.1))

# Load the specific dataset
dataset_name, dataset_path, dataset_specification, eval_percent, eval_method = datasets[0]
ft.load_dataset(dataset_name, dataset_path, dataset_specification, evaluation_dataset_sampling=eval_method, evaluation_percent=eval_percent)


Using cache file path: cache/Balanced_Dataset_Binary_0_QdLmZHuh8yOmlGcKBEkf7hepImY0_A6N00gtYIhwW1x05bzV0RseOHrU0.feather
Reading directly from cache cache/Balanced_Dataset_Binary_0_QdLmZHuh8yOmlGcKBEkf7hepImY0_A6N00gtYIhwW1x05bzV0RseOHrU0.feather...


,OUT_BYTES,IN_PKTS,OUT_PKTS,IN_BYTES,FLOW_DURATION_MILLISECONDS,L4_SRC_PORT_1,L4_SRC_PORT_2,L4_SRC_PORT_3,L4_SRC_PORT_4,L4_SRC_PORT_5,...,TCP_FLAGS_8,TCP_FLAGS_9,TCP_FLAGS_10,TCP_FLAGS_11,TCP_FLAGS_12,TCP_FLAGS_13,TCP_FLAGS_14,TCP_FLAGS_15,TCP_FLAGS_16,TCP_FLAGS_17
0,0.561348,0.173847,0.219600,0.371261,0.710869,True,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False
1,0.122540,0.000000,0.000000,0.000000,0.000000,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,0.122540,0.000000,0.000000,0.000000,0.000000,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,0.122540,0.000000,0.000000,0.000000,0.000000,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,0.264783,0.091847,0.059344,0.246059,0.000000,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,0.304556,0.000000,0.000000,0.000000,0.000000,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
19996,0.351373,0.134554,0.137793,0.288161,0.452565,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
19997,0.122540,0.000000,0.000000,0.000000,0.000000,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
19998,0.122540,0.000000,0.000000,0.000000,0.000000,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [5]:
m = ft.build_model()
m.summary()


2026-04-12 13:51:55.787400: E tensorflow/stream_executor/cuda/cuda_driver.cc:271] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2026-04-12 13:51:55.787424: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (kali): /proc/driver/nvidia/version does not exist
2026-04-12 13:51:55.788073: I tensorflow/core/platform/cpu_feature_guard.cc:151] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_OUT_BYTES (InputLayer)   [(None, 8, 1)]       0           []                               
                                                                                                  
 input_IN_PKTS (InputLayer)     [(None, 8, 1)]       0           []                               
                                                                                                  
 input_OUT_PKTS (InputLayer)    [(None, 8, 1)]       0           []                               
                                                                                                  
 input_IN_BYTES (InputLayer)    [(None, 8, 1)]       0           []                               
                                                                                              

In [6]:
m.compile(optimizer="adam", loss='binary_crossentropy', metrics=['binary_accuracy'], jit_compile=True)

# Get the evaluation results
eval_results: pd.DataFrame
(train_results, eval_results, final_epoch) = ft.evaluate(m, batch_size=128, epochs=5, steps_per_epoch=64, early_stopping_patience=5)

print(eval_results)

Building eval dataset...
Splitting dataset to featurewise...
Evaluation dataset is built!
Positive samples in eval set: 1952
Negative samples in eval set: 2048


2026-04-12 13:51:57.648776: I tensorflow/compiler/xla/service/service.cc:171] XLA service 0x7fdb2c050ba0 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
2026-04-12 13:51:57.648802: I tensorflow/compiler/xla/service/service.cc:179]   StreamExecutor device (0): Host, Default Version
2026-04-12 13:51:57.705796: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:237] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-04-12 13:51:57.712454: W tensorflow/compiler/tf2xla/kernels/random_ops.cc:57] Warning: Using tf.random.uniform with XLA compilation will ignore seeds; consider using tf.random.stateless_uniform instead if reproducible behavior is desired. model/block_0_transformer_encoder/block_0_attention_dropout/dropout/random_uniform/RandomUniform
2026-04-12 13:52:04.445791: I tensorflow/compiler/jit/xla_compilation_cache.cc:399] Compiled cluster using XLA!  This line is logged at most once for th

Epoch = 0 / 5 (early stop in 5), step = 0, loss = 0.78017, results = [0.7801719903945923, 0.5546875] -- elapsed (train): 0.00s
Epoch = 1 / 5 (early stop in 5), step = 4, loss = 0.05325, results = [0.05324924737215042, 0.9921875] -- elapsed (train): 2.40s
Epoch = 2 / 5 (early stop in 4), step = 9, loss = 0.00558, results = [0.005583730526268482, 1.0] -- elapsed (train): 4.82s
Epoch = 3 / 5 (early stop in 5), step = 14, loss = 0.03130, results = [0.03129514306783676, 0.9921875] -- elapsed (train): 7.20s
Epoch = 4 / 5 (early stop in 4), step = 19, loss = 0.00899, results = [0.008987762033939362, 1.0] -- elapsed (train): 9.58s
125/125 [==============================] - 1s 5ms/step
Epoch 4 yielded predictions: (4000,), overall balanced accuracy: 99.60%, TP = 1,947 / 1,952, TN = 2,037 / 2,048
   epoch     P     N  pred_P  pred_N    TP  FP    TN  FN   bal_acc        f1
0      4  1952  2048    1958    2042  1947  11  2037   5  0.996034  0.995908
